# Matched three-pipeline training

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from manuscript_analysis import find_project_root
from manuscript_analysis.workflows import canonical

ROOT = find_project_root()
STATE = canonical.setup(ROOT)
display(STATE["master"].groupby(["class_name", "class_label"]).size())

[UNSORTED] reading /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/input/unsorted/CTRL-CO-DAY150.pkl
[UNSORTED] reading /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/input/unsorted/AD-CO-DAY150.pkl
[SORTED] reading /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/input/sorted/CTRL-CO-DAY150.pkl
[SORTED] reading /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/input/sorted/AD-CO-DAY150.pkl
[PRUNED] reading /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/input/pruned/CTRL-CO-DAY150.pkl
[PRUNED] reading /Users/cuddysteadman/Desktop/DBC/manuscript-analysis/input/pruned/AD-CO-DAY150.pkl


class_name  class_label
AD-CO       1              31
CTRL-CO     0              19
dtype: int64

## Train or resume all pipeline/seed combinations

In [2]:
canonical.train(STATE)


[23/25] seed 22
  UNSORTED  ✓  13 epochs · 85s · session AUC 0.929
  SORTED    ✓  11 epochs · 40s · session AUC 0.733
  PRUNED    ✓  8 epochs · 52s · session AUC 0.964

[24/25] seed 23
  UNSORTED  ✓  8 epochs · 50s · session AUC 0.964
  SORTED    ✓  6 epochs · 21s · session AUC 1.000
  PRUNED    ✓  13 epochs · 69s · session AUC 0.893

[25/25] seed 24
  UNSORTED  ✓  5 epochs · 35s · session AUC 1.000
  SORTED    ✓  10 epochs · 38s · session AUC 1.000
  PRUNED    ✓  7 epochs · 44s · session AUC 0.964


## Consolidate and audit saved artifacts

In [5]:
RESULTS = canonical.consolidate(STATE)
display(RESULTS["run_summary"])
display(RESULTS["pooled_confusions"])

,pipeline,n_runs,session_auc_mean,session_auc_sd,session_accuracy_mean,block_auc_mean,train_time_total_s
0,PRUNED,25,0.942857,0.047246,0.847273,0.885064,1182.623
1,SORTED,25,0.877333,0.122746,0.770000,0.774597,698.741
2,UNSORTED,25,0.964286,0.034194,0.876364,0.900901,1182.579


,pipeline,level,threshold_mode,tn,fp,fn,tp,n_total
0,PRUNED,block,tuned,12058,3359,5809,27067,48293
1,PRUNED,block,naive,11206,4211,4515,28361,48293
2,PRUNED,session,tuned,85,15,27,148,275
3,PRUNED,session,naive,86,14,24,151,275
4,SORTED,block,tuned,6456,2285,5635,7168,21544
5,SORTED,block,naive,5551,3190,4310,8493,21544
6,SORTED,session,tuned,56,19,27,98,200
7,SORTED,session,naive,55,20,26,99,200
8,UNSORTED,block,tuned,13699,3378,5068,30397,52542
9,UNSORTED,block,naive,13595,3482,4383,31082,52542


In [10]:
auc_summary = (
    RESULTS["run_metrics"]
    .groupby("pipeline")["block_auc"]
    .agg(
        n="count",
        median="median",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
    )
)

auc_summary["median [Q1, Q3] AUC"] = auc_summary.apply(
    lambda row: f'{row["median"]:.3f} [{row["q1"]:.3f}, {row["q3"]:.3f}]',
    axis=1,
)

auc_summary[["n", "median [Q1, Q3] AUC"]]

,n,"median [Q1, Q3] AUC"
pipeline,,
PRUNED,25,"0.894 [0.819, 0.946]"
SORTED,25,"0.850 [0.686, 0.882]"
UNSORTED,25,"0.906 [0.857, 0.955]"


In [12]:
import pandas as pd
from scipy.stats import wilcoxon

auc_by_seed = (
    RESULTS["run_metrics"]
    .pivot(index="seed", columns="pipeline", values="block_auc")
)

comparisons = [
    ("UNSORTED", "SORTED"),
    ("UNSORTED", "PRUNED"),
    ("PRUNED", "SORTED"),
]

pairwise_tests = []
for first, second in comparisons:
    paired = auc_by_seed[[first, second]].dropna()
    test = wilcoxon(
        paired[first],
        paired[second],
        alternative="two-sided",
        zero_method="wilcox",
    )
    pairwise_tests.append({
        "comparison": f"{first} vs {second}",
        "n_pairs": len(paired),
        "p_raw": test.pvalue,
    })

pairwise_tests = pd.DataFrame(pairwise_tests)
pairwise_tests["p_raw"] = pairwise_tests["p_raw"].map(lambda p: f"{p:.3g}")
pairwise_tests

,comparison,n_pairs,p_raw
0,UNSORTED vs SORTED,25,0.000287
1,UNSORTED vs PRUNED,25,0.071
2,PRUNED vs SORTED,25,0.000631
